In [ ]:
# !pip install TCT

Link to the TCT document: https://ncatstranslator.github.io/Translator_component_toolkit/


In [ ]:

from TCT import translator_metakg
from TCT import translator_kpinfo
import matplotlib.pyplot as plt
import networkx as nx

In [ ]:
def load_translator_resources():
    """
    Load the necessary resources for the Translator.
    """
    Translator_KP_info,APInames= translator_kpinfo.get_translator_kp_info()
    metaKG = translator_metakg.get_KP_metadata(APInames) 
    APInames,metaKG = translator_metakg.add_plover_API(APInames, metaKG)
    return  APInames, metaKG, Translator_KP_info

APInames, metaKG, Translator_KP_info= load_translator_resources()

In [ ]:
# Preparation 
# Step1: List all the APIs in the translator system
Translator_KP_info,APInames= translator_kpinfo.get_translator_kp_info()
print(len(Translator_KP_info))
# Step 2: Get metaKG and all predicates from Translator APIs through the SmartAPI system
metaKG = translator_metakg.get_KP_metadata(APInames) 
print(metaKG.shape)
# Add metaKG from Plover API based KG resources
APInames,metaKG = translator_metakg.add_plover_API(APInames, metaKG)
print(metaKG.shape)
# Step 3: list metaKG information
All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))
print(len(API_withMetaKG))
print(len(All_predicates))
print(len(All_categories))

# ARA list
API_withMetaKG = set(metaKG['API'])
print("ARA list:", set(APInames.keys()) - API_withMetaKG)

In [ ]:
APInames

In [ ]:
metaKG

Understand the metaKG in biolink: https://biolink.github.io/biolink-model/categories.html

In [ ]:
# find the KG in one individual API (optional)
metaKG.loc[metaKG['API'] == 'Drug Approvals KP - TRAPI 1.5.0',['API','Predicate','Subject','Object']].drop_duplicates()

In [ ]:
# find the KG in one individual API (optional)
metaKG.loc[metaKG['API'] == 'MolePro',['API','Predicate','Subject','Object']].drop_duplicates()

In [ ]:

# find the KG in one individual API (optional)
metaKG.loc[metaKG['API'] == 'RTX KG2 - TRAPI 1.5.0',['API','Predicate','Subject','Object']].drop_duplicates()

In [ ]:
# draw the interaction graph between subject and object in the metaKG using networkx
# first, we need to drop the interactions between subjects and objects that both subjects and objects are the same
#metaKG = metaKG[metaKG['Subject'] != metaKG['Object']]
# second, we need to filter the metaKG to only include the selected categories
#metaKG = metaKG[metaKG['Subject'].isin(selected_categories) | metaKG['Object'].isin(selected_categories)]
selected_KGs = ['Drug Approvals KP - TRAPI 1.5.0'
                ]

metaKG_sele = metaKG[metaKG['API'].isin(selected_KGs)]

# build a multigraph to capture all edges (including duplicates) and their predicates
G = nx.MultiGraph()
for _, row in metaKG_sele.iterrows():
        G.add_edge(row['Subject'], row['Object'], predicate=row['Predicate'])

# layout and draw nodes + edges
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5, iterations=20)
nx.draw(G, pos,
                with_labels=True,
                node_size=50,
                font_size=12,
                font_color='black',
                node_color='blue',
                edge_color='gray')

# draw edge labels
edge_labels = nx.get_edge_attributes(G, 'predicate')
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)
plt.title('Interaction Graph of Subjects and Objects in metaKG')
plt.show()